# The MLflow Prompt Optimization Loop

A guided workflow for iteratively improving LLM prompts using MLflow as your observability and evaluation platform.

```
Create Prompt  →  Run Queries  →  Add User Feedback
      ↑                                   ↓
 Optimized Prompt  ←  Evaluate  ←  Upload Eval Dataset
```

**What you'll do in this notebook:**
1. **Create** a system prompt and register it in MLflow's Prompt Registry
2. **Run** queries — with traces automatically linked to the prompt version
3. **Add** simulated user feedback to each trace
4. **Upload** an evaluation dataset to MLflow
5. **Evaluate** your prompt against expectations
6. **Optimize** — let MLflow automatically improve the prompt using `optimize_prompts`
7. **Re-evaluate** to measure the improvement

---
## 0. Setup

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

try:
    from dotenv import load_dotenv
    load_dotenv(override=True)
except ImportError:
    pass

import mlflow
from mlflow.entities import AssessmentSource, AssessmentSourceType
from mlflow.genai.scorers import scorer
from mlflow.genai.optimize import GepaPromptOptimizer
from openai import OpenAI
import pandas as pd

from support_functions import get_namespace

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CONFIGURATION — update these to match your environment
# ─────────────────────────────────────────────────────────────────

# MLflow server
MLFLOW_TRACKING_URI = "https://rh-ai.<CLUSTER-DOMAIN>/mlflow"
MLFLOW_TRACKING_TOKEN = "<OPENSHIFT-API-TOKEN>"

# MLflow names (change if you want to)
EXPERIMENT_NAME = "prompt-optimization-loop"
PROMPT_NAME = "support-assistant-prompt"
DATASET_NAME = "support-qa-eval"

# LLM for inference
LLM_API_KEY = "<MODEL-API-KEY>"
LLM_BASE_URL = "<MODEL-ENDPOINT>"
LLM_MODEL = "<MODEL-NAME>"

In [ ]:
# Connect to MLflow
os.environ["MLFLOW_TRACKING_TOKEN"] = MLFLOW_TRACKING_TOKEN
os.environ["MLFLOW_WORKSPACE"] = get_namespace()
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] =="true"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

---
## Step 1: Create Your Prompt

Write a system prompt and register it in **MLflow's Prompt Registry**.  
Every call to `register_prompt` creates a new immutable version — you can always diff, reload, or roll back to any previous version.

In [ ]:
# Intentionally minimal — gives the optimizer room to improve it
SYSTEM_PROMPT_V1 = """You are a helpful customer support assistant for Acme Software.
Answer customer questions clearly and accurately."""

prompt_v1 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=SYSTEM_PROMPT_V1,
    commit_message="v1: baseline prompt",
    tags={"version_type": "baseline"},
)

PROMPT_V1_URI = f"prompts:/{PROMPT_NAME}/{prompt_v1.version}"

print(f"✓ Registered '{PROMPT_NAME}' as version {prompt_v1.version}")

---
## Step 2: Run Queries

We connect the traces to the prompt by loading the prompt we just created.  
This way we can easily filter what conversations happened with which prompt version.

In [ ]:
# Sample queries — simulate real user traffic
SAMPLE_QUERIES = [
    "How do I reset my password?",
    "What is the refund policy?",
    "How do I export my data?",
    "Is there a mobile app?",
    "How do I cancel my subscription?",
]

# Enable MLflow tracing for OpenAI calls (adds child spans inside any @mlflow.trace)
mlflow.openai.autolog()

llm_client = OpenAI(
    api_key=LLM_API_KEY,
    **(dict(base_url=LLM_BASE_URL) if LLM_BASE_URL else {}),
)

_last_trace_id: list = []

@mlflow.trace(name="ask")
def ask_traced(question: str, prompt_uri: str) -> str:
    """
    Call the LLM with a prompt loaded from the registry.
    - load_prompt() INSIDE @mlflow.trace links this trace to the prompt version.
    - get_active_trace_id() is captured here (only valid while trace is active)
      and stored in _last_trace_id so the caller can attach feedback.
    """
    _last_trace_id.clear()
    _last_trace_id.append(mlflow.get_active_trace_id())  # ← capture before returning

    prompt = mlflow.genai.load_prompt(prompt_uri)        # ← prompt linkage

    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": prompt.template},
            {"role": "user",   "content": question},
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content

In [ ]:
# Now we can run the queries

v1_responses = []  # stores {question, answer, trace_id} — used for feedback in next step

for i, question in enumerate(SAMPLE_QUERIES, 1):
    answer = ask_traced(question, PROMPT_V1_URI)

    trace_id = _last_trace_id[0] if _last_trace_id else None

    v1_responses.append({"question": question, "answer": answer, "trace_id": trace_id})

    print(f"[{i}] Q: {question}\n")
    print(f"A: {answer}\n")
    print(f"trace_id: {trace_id}\n")

Go to MLFlow to see the traces (conversations)

In [ ]:
MLFLOW_TRACKING_URI

---
## Step 3: Add User Feedback to Traces

`mlflow.log_feedback()` attaches structured feedback directly to a trace — thumbs up/down, a score, or free text.

In production this would come from your UI (e.g. a 👍/👎 button next to each response). Here we simulate it. The feedback is visible in the MLflow Traces tab alongside the trace details and can be used to filter or prioritise examples for evaluation.

In [ ]:
SIMULATED_FEEDBACK = [
    {"rating": True,  "comment": "Clear and easy to follow"},
    {"rating": False, "comment": "Too vague — didn't explain the actual policy"},
    {"rating": True,  "comment": "Helpful, answered my question"},
    {"rating": True,  "comment": "Short and to the point"},
    {"rating": False, "comment": "Didn't tell me where to go to cancel"},
]

print("Attaching user feedback to traces...\n")

for resp, fb in zip(v1_responses, SIMULATED_FEEDBACK):
    trace_id = resp["trace_id"]
    if not trace_id:
        print(f"  ⚠ No trace ID captured for: {resp['question'][:40]}")
        continue

    mlflow.log_feedback(
        trace_id=trace_id,
        name="user_satisfaction",
        value=fb["rating"],
        rationale=fb["comment"],
        source=AssessmentSource(
            source_type=AssessmentSourceType.HUMAN,
            source_id="demo_user",
        ),
    )

    icon = "👍" if fb["rating"] else "👎"
    print(f"  {icon}  {resp['question']}")
    print(f"       '{fb['comment']}'")
    print()

print("✓ Feedback attached!")
print(f"  → Open a trace in the MLflow UI to see the feedback panel.")

In [ ]:
# Expectations — what a correct answer must satisfy for each query
# Stored on the traces so the dataset can be built from them in the next step
EXPECTATIONS = [
    {"must_contain": "password", "max_chars": 200},
    {"must_contain": "refund",   "max_chars": 200},
    {"must_contain": "export",   "max_chars": 200},
    {"must_contain": "app",      "max_chars": 200},
    {"must_contain": "cancel",   "max_chars": 200},
]

print("Attaching expectations to traces...\n")

for resp, exp in zip(v1_responses, EXPECTATIONS):
    trace_id = resp["trace_id"]
    if not trace_id:
        print(f"  ⚠ No trace ID captured for: {resp['question'][:40]}")
        continue

    mlflow.log_feedback(
        trace_id=trace_id,
        name="expected_keyword",
        value=exp["must_contain"],
        rationale=f"Response must mention '{exp['must_contain']}'",
        source=AssessmentSource(
            source_type=AssessmentSourceType.HUMAN,
            source_id="annotator",
        ),
    )
    mlflow.log_feedback(
        trace_id=trace_id,
        name="expected_max_chars",
        value=str(exp["max_chars"]),
        rationale=f"Response must be under {exp['max_chars']} characters",
        source=AssessmentSource(
            source_type=AssessmentSourceType.HUMAN,
            source_id="annotator",
        ),
    )

    print(f"  [{exp['must_contain']:<10} ≤{exp['max_chars']}ch]  {resp['question']}")

print("\n✓ Expectations attached to traces!")

Go to MLFlow UI again and see the feedback inside one of the traces 🎉

---
## Step 4: Upload the Expectation Dataset to MLflow

Now that we have some feedback, we can add expectations to the traces.  
The expectations just say what we expected from the output of each conversation.  
After we have a few expectations, we can add those traces to a dataset which we later will use for evaluating current and future conversations :D

You can add expectations and create the dataset through the UI (inside each trace), but here we do it by uploading a dataset right away.

In [ ]:
# Retrieve the traces we just annotated
our_trace_ids = {r["trace_id"] for r in v1_responses if r["trace_id"]}

traces_df = mlflow.search_traces(
    experiment_ids=[experiment.experiment_id],
    max_results=50,
)

# Filter to only the traces from our query run (there may be child spans from autolog too)
annotated_traces = traces_df[traces_df["request_id"].isin(our_trace_ids)]
print(f"Found {len(annotated_traces)} annotated traces")

# Build dataset records: inputs/outputs sourced from the traces,
# expectations reconstructed from the annotations we attached above.
# (In a real system you'd parse these from the trace assessments returned in traces_df)
records = [
    {
        "inputs":       {"question": resp["question"]},
        "outputs":      resp["answer"],
        "expectations": exp,
    }
    for resp, exp in zip(v1_responses, EXPECTATIONS)
]

eval_dataset = mlflow.genai.create_dataset(
    name=DATASET_NAME,
    experiment_id=experiment.experiment_id,
    tags={"source": "annotated-traces", "version": "1"},
)
eval_dataset.merge_records(records)

print(f"✓ Dataset '{DATASET_NAME}' created from {len(records)} annotated traces")
print(f"  → Browse it in the MLflow UI: {MLFLOW_TRACKING_URI}")

---
## Step 5: Evaluate Prompt v1

`mlflow.genai.evaluate()` runs every record in the dataset through the model and scores the output.

We pass the **MLflow-hosted dataset** (not the local list) — this links the evaluation run to the dataset in the UI so you can trace which data produced which results.

In [ ]:
@scorer
def contains_keyword(inputs: dict, outputs: str, expectations: dict) -> bool:
    """Does the response contain the required keyword?"""
    if not outputs or not expectations:
        return False
    return expectations.get("must_contain", "").lower() in outputs.lower()


@scorer
def is_concise(outputs: str, expectations: dict) -> bool:
    """Is the response under the character limit?"""
    if not outputs:
        return False
    return len(outputs) <= expectations.get("max_chars", 300)


print("✓ Scorers defined: contains_keyword, is_concise")

In [ ]:
# predict_fn — parameter names must match the "inputs" keys in the dataset
def predict_v1(question: str) -> str:
    return ask_traced(question, PROMPT_V1_URI)


print("Running evaluation with prompt v1...")
eval_v1 = mlflow.genai.evaluate(
    data=eval_dataset,      # ← using the MLflow-hosted dataset
    predict_fn=predict_v1,
    scorers=[contains_keyword, is_concise],
)

print("\n── Evaluation Results: Prompt v1 ──")
for metric, value in sorted(eval_v1.metrics.items()):
    bar = "█" * int((value if isinstance(value, float) else 0) * 20)
    print(f"  {metric:<35} {f'{value:.0%}' if isinstance(value, float) else value}  {bar}")

print(f"\n✓ Results saved. View per-example breakdowns in the Evaluations tab.")
print(f"  → {MLFLOW_TRACKING_URI}")

---
## Step 6: Optimize the Prompt with MLflow

Now that we have a expectation dataset and have evaluated how well our current prompt works, we can improve the prompt.  
The naive way to do this would be manually, but that takes a lot of time and effort.  
Instead, we will programatically improve the prompt using MLFlows prompt optimizer.

Instead of manually rewriting the prompt, we use `mlflow.genai.optimize_prompts()`, which uses something called GEPA.

**How it works (GEPA — Generate, Evaluate, Predict, Adapt):**
1. Runs the `predict_fn` on the training data to collect initial scores
2. Uses the `reflection_model` to analyse which examples failed and why (typically a different model but in this case we use the same model as for normal predictions)
3. Generates a rewritten prompt that addresses the identified weaknesses
4. Registers the improved prompt as a new version in the Prompt Registry

In [ ]:
if LLM_BASE_URL:
    os.environ["OPENAI_BASE_URL"] = LLM_BASE_URL
os.environ["OPENAI_API_KEY"] = LLM_API_KEY


def predict_fn(question: str) -> str:
    """
    Predict function for the optimizer.
    IMPORTANT: mlflow.genai.load_prompt() must be called here — the optimizer
    intercepts this call to identify which prompt template to rewrite.
    """
    prompt = mlflow.genai.load_prompt(PROMPT_V1_URI)   # ← optimizer hooks in here
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": prompt.template},
            {"role": "user",   "content": question},
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content


print(f"Optimizing '{PROMPT_NAME}' using openai:/{LLM_MODEL} as the reflection model...")
print(f"This may take a few minutes as the optimizer runs multiple evaluation passes.\n")

opt_result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=eval_dataset,          # same dataset we uploaded
    prompt_uris=[PROMPT_V1_URI],
    optimizer=GepaPromptOptimizer(reflection_model=f"openai:/{LLM_MODEL}"),
    scorers=[contains_keyword, is_concise],
)

OPTIMIZED_PROMPT_URI = opt_result.optimized_prompts[0].uri

print(f"\n✓ Optimization complete!")

optimized_prompt = mlflow.genai.load_prompt(OPTIMIZED_PROMPT_URI)
print(f"\nOptimized prompt template:\n{'─'*50}")
print(optimized_prompt.template)
print("─"*50)

You can now go and view the optimized prompt as a new prompt version inside of MLFlow Dashboard 📊

---
## Step 7: Evaluate the Optimized Prompt & Compare

Run the **same evaluation dataset** through the optimized prompt and compare scores side-by-side.  
Because the dataset, scorers, and questions are identical, any score difference is purely attributable to the prompt change.

In [ ]:
def predict_optimized(question: str) -> str:
    return ask_traced(question, OPTIMIZED_PROMPT_URI)


print("Running evaluation with optimized prompt...")
eval_optimized = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_optimized,
    scorers=[contains_keyword, is_concise],
)

print("\n── Evaluation Results: Optimized Prompt ──")
for metric, value in sorted(eval_optimized.metrics.items()):
    bar = "█" * int((value if isinstance(value, float) else 0) * 20)
    print(f"  {metric:<35} {f'{value:.0%}' if isinstance(value, float) else value}  {bar}")

In [ ]:
# Side-by-side comparison table
print("\n══════════════════════════════════════════════════")
print("      COMPARISON: v1 (baseline) vs optimized      ")
print("══════════════════════════════════════════════════")

rows = []
all_metrics = sorted(set(eval_v1.metrics) | set(eval_optimized.metrics))
for metric in all_metrics:
    v1_val  = eval_v1.metrics.get(metric)
    opt_val = eval_optimized.metrics.get(metric)
    if isinstance(v1_val, float) and isinstance(opt_val, float):
        delta  = opt_val - v1_val
        change = f"+{delta:.0%}" if delta > 0 else (f"{delta:.0%}" if delta < 0 else "no change")
        symbol = "✅" if delta > 0 else ("❌" if delta < 0 else "  ")
        rows.append({
            "Metric":     metric,
            "v1":         f"{v1_val:.0%}",
            "optimized":  f"{opt_val:.0%}",
            "Change":     change,
            " ":          symbol,
        })

if rows:
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("No numeric metrics found — check the Evaluations tab in MLflow.")

You can also compare the evaluation results side-by-side in MLFlow Dashboard